### Project Ariadne Delta Logprob Datasets

To measure the correlation between the ID-deltas and the OOD-deltas, we've got to create both datasets
from the samples we've collected in the previous step. 

For the OOD-Data, we've collected ground truth samples from the Qwen2.5-7B basemodel and augmented them using Qwen3-8B. For the ID-Data, we've collected rollouts for each checkpoint $i \in \{0, 10, 20, ..., 80\}$ and distilled answers for which Qwen3-8B decided they contain a calculation error. This gives us pairs $(x_{\text{gt}}, x_{\text{aug}})$ for the OOD-data and pairs $(x_{\text{gt}}, x_{\text{error}})$ for the ID-samples.

The question is, whether $\rho = \text{corr}(\Delta_{\text{ID}}, \Delta_{\text{OOD}}) > 0$ 

In [1]:
import os
import numpy as np
import pandas as pd

In [111]:
with open('/u/rfechner/data/ariadne/id-outputs-simple-qw3-8b.parquet', 'rb') as file:
    df_id = pd.read_parquet(file)

In [112]:
len(df_id)

11062

In [113]:
df_id.iloc[0]['responses'][0]

"<think>\nOkay, let's see. The student's answer is p - q + 1/8, but the correct answer is p - q. So where did the student go wrong?\n\nLooking at their steps, they transformed the double sum into a single sum over n from 2 to infinity of (1/n² - 1/n³). Then they split that into two sums: sum 1/n² from 2 to ∞ minus sum 1/n³ from 2 to ∞. \n\nFor the first sum, they said it's p - 1. That makes sense because p is the sum from 1 to ∞, so subtracting the first term (1/1²) gives the sum from 2 to ∞. \n\nFor the second sum, they said it's q - 1 - 1/8. Wait, q is the sum from 1 to ∞ of 1/k³. So subtracting the first term (1/1³) would give the sum from 2 to ∞. But the student subtracted 1 and 1/8. Wait, 1/8 is 1/2³. So they subtracted the first term (1) and the second term (1/8)? But that's not correct. The sum from n=2 to ∞ of 1/n³ is q minus the first term (n=1), which is 1. The student subtracted 1 and 1/8, which would be subtracting the first two terms. But the original sum starts at n=2, so

In [114]:
# filter id dataset
def filter_in_distribution(df : pd.DataFrame) -> pd.DataFrame:
    """
        Parses and filters dataset.

        Responses of the in-distribution dataframe are of the shape:
        <think>...</think> ... #### {yes|no} ...
        We should filter out responses which contain a "#### yes"
    """
    def mapper(response : list[str]):
        r = response[0][-100:]
        try:
            i = r.rindex('####')
        except ValueError:
            return False
        return 'yes' in r[i:i+10]
        
    cond = df['responses'].apply(mapper)

    return df[cond]
df_id_filtered = filter_in_distribution(df_id)

In [115]:
len(df_id_filtered)

4576

In [116]:
# we have duplicates, as the old index refers to the old index **per-checkpoint**
len(df_id_filtered.old_index), len(set(df_id_filtered.old_index)) 

(4576, 4423)

In [117]:
# load original dataset the id-base-dataset was constructed from;
# use /u/rfechner/verl/workspace/dataset_generation/create_ariadne_id_base_dataset_simpleprompt.ipynb
path = "/ptmp/rfechner/out/exp05_rollouts_qwen2.5-7b/qwen2.5_7b__gspo/val_jsonl"
checkpoints = []
for file in os.listdir(path):
    with open(os.path.join(path, file), 'r') as jsonfile:
        cp = int(file.split('_')[0])
        df = pd.read_json(jsonfile, lines=True)
        df['checkpoint'] = [cp] * len(df)
        checkpoints.append(df)

In [118]:
def filter_nonzero_std_and_onlymath500(dfs : list[pd.DataFrame]) -> list[pd.DataFrame]:
    """
        Filters a list of dataframes to only include samples which have non-zero std
        rewards -> which are solved and not solved in the same set of answers.
    """
    ret = []
    for df in dfs:
        df = df.iloc[:500 * 256]
        mask = df.groupby('input', sort=False)['score'].transform(lambda x: x.std() != 0)
        candidates = df[mask]
        ret.append(candidates)
    return ret

dfs = filter_nonzero_std_and_onlymath500(checkpoints)
# double check whether this equals the original dataset lengths: [51200, 118528, 56832, 83712, 63232, 115968, 69120, 56832, 59648]
[len(df) for df in dfs] 

[51200, 118528, 56832, 83712, 63232, 115968, 69120, 56832, 59648]

In [119]:
import warnings
from functools import wraps
def ignore_warnings(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return func(*args, **kwargs)
    return wrapper

def filter_overlong_sequences(dfs: list[pd.DataFrame], max_resp_len : int = 1024) -> list[pd.DataFrame]:
    ret = []
    for df in dfs:
        approx_tokens = (df['input'].apply(len) + df['output'].apply(len)) / 3
        ret.append(df[approx_tokens < max_resp_len])
    return ret

@ignore_warnings
def filter_negative_samples(dfs : list[pd.DataFrame], keep_ans_per_q : int = 5) -> list[pd.DataFrame]:
    """
        Given a list of pandas DataFrames this function returns the same dataframes, but
        only questions which are negatively answered.

        It retains the initial question index and row index of the response to later match
        back.

        Also: we're filtering out sequences which contain '```python' or do not contain '\\boxed{..}' within the
        last 300 characters.
    """
    ret = []
    for df in dfs:
        negdf = df[df['score'] <= 0]

        cond = negdf['output'].apply(lambda x: "python" not in x and '\\boxed' in x[-100:])
        negdf = negdf[cond]
        sampled = (
            negdf.groupby("input", sort=False, group_keys=False)
            .apply(lambda g: g.sample(n=min(len(g), keep_ans_per_q), random_state=0))
        )
        ret.append(sampled)
    return ret

In [120]:
negatives = filter_negative_samples(filter_overlong_sequences(dfs))
# should [825, 1674, 996, 1498, 1096, 1760, 1227, 938, 1048]
list(map(len, negatives))

[825, 1674, 996, 1498, 1096, 1760, 1227, 938, 1048]

In [121]:
negatives = pd.concat(negatives, axis=0)
negatives.head(2)

,input,output,gts,score,step,reward,acc,checkpoint
364,system\nYou are a helpful assistant.\nuser\nDe...,To find a way to write the double sum \(\sum_{...,p - q,0,80,0,0,80
2431,system\nYou are a helpful assistant.\nuser\nTh...,To determine how many different values can be ...,4,0,80,0,0,80


In [122]:
df_id_filtered.head(2)

,prompt,step,old_index,responses
0,[{'content': 'You are a helpful judge and an e...,80,364,"[<think>\nOkay, let's see. The student's answe..."
18,[{'content': 'You are a helpful judge and an e...,80,4023,"[<think>\nOkay, let's try to figure out why th..."


In [123]:
# for each answer in df_id_filtered, we've decided to treat it as answer that contains a calculation error. we need to map
# that sample back to the question (input), then map from input to ground truth answer. The dfs should contain
# the inputs which have a ground truth in-distribution response. I've really made this hard on myself..

new_negatives = []
neg_groups = negatives.groupby('step', sort=False)
for step, df in df_id_filtered.groupby('step', sort=False):
    tmp_df = neg_groups.get_group(step)
    new_negatives.append(tmp_df.loc[df['old_index']])
negatives_filt = pd.concat(new_negatives, axis=0)

In [124]:
all_dfs = pd.concat(dfs, axis=0)
all_dfs_groups = all_dfs.groupby('step', sort=False)
counter = 0
rows = []

for ind, df in negatives_filt.groupby('step', sort=True):
    adf = all_dfs_groups.get_group(ind)
    adf_question_groups = adf.groupby('input', sort=False)
    for q, df_per_question in df.groupby('input', sort=False):
        """
            Per-checkpoint, per question with nonzero std in scores,
            we may have multiple correct and incorrect answers. I'm going to
            assume that the N=5 correct answers are as good as any. For each wrong
            answer in the df_per_question dataframe, we're going to add a single row to the dataframe.

            Hence, for each checkpoint, for each question, we're getting N anchor responses and M wrong responses.
        """
        tmp = adf_question_groups.get_group(q)
        correct_answers = tmp[tmp['score'] > 0]['output'][:5]
        question = q.removeprefix('system\nYou are a helpful assistant.\nuser\n').removesuffix("\nassistant\n")
        rows.extend([{ # rows for ground truths

            'prompt' : [{'role' : 'user', 'content' : question}, 
                        {'role' : 'assistant', 'content' : correct}],
            'behaviour_type' : 'anchor',
            'step' : ind,
            "uid" : counter} for correct in correct_answers])
        
        rows.extend([{ # rows for calculation errors

            'prompt' : [{'role' : 'user', 'content' : question}, 
                        {'role' : 'assistant', 'content' : output}],
            'behaviour_type' : 'calculation_error',
            'step' : ind,
            "uid" : counter
        } for output in df_per_question['output']])
        counter += 1

In [127]:
# at the time of writing this, I have to make the number of lines in the deltas dataframe I'm passing divisible by 8.
# It is what it is.
id_dataframe = pd.DataFrame(rows)
len(id_dataframe)

12842

In [128]:
drop_last=len(id_dataframe)%8
id_dataframe=id_dataframe.iloc[:-drop_last]
assert len(id_dataframe)%8==0
len(id_dataframe)

12840

In [129]:
path = "/u/rfechner/data/ariadne"
os.makedirs(path, exist_ok=True)

with open(os.path.join(path, 'id-deltas.jsonl'), 'w') as file:
    id_dataframe.to_json(path_or_buf=file, lines=True, orient='records')

#### OOD Answers

Let's load in the generated answers, group them to the original ground truths and then create the ood-deltas dataset.

In [ ]:
with open('/u/rfechner/data/ariadne/debug-ood-outputs-simple2-longeranswers.parquet', 'rb') as file:
    df_ood_gen = pd.read_parquet(file)

In [79]:
len(df_ood_gen)

1296

In [58]:
df_ood_gen.head(1)

,prompt,step,old_index,responses
132,[{'content': 'You are a helpful case generator...,0,132,"[<think>\nOkay, let's see. The user wants me t..."


In [59]:
df_ood_gen.iloc[0]['responses'][0]

"<think>\nOkay, let's see. The user wants me to take the correct answer and inject an arithmetic or algebraic error, but only if it's reasonable. The original solution converts (0,3) to polar coordinates. The correct steps are calculating r as the distance, which is sqrt(0² + 3²) = 3, and theta as arctan(y/x), but since x is 0, it's pi/2. \n\nSo the correct answer is (3, pi/2). Now, how can I introduce an error here? Let me think. Maybe the student could have miscalculated r. For example, if they forgot to square the coordinates or added instead of squaring. Wait, if they did 0 + 3 instead of sqrt(0² + 3²), they might get 3, which is correct. Hmm. Alternatively, maybe they thought r is just y, which is 3, but that's actually correct. So maybe that's not an error.\n\nWait, what if they miscalculated theta? The correct theta is pi/2. Suppose they thought that since it's on the y-axis, theta is pi/4? But that's not right. Or maybe they did arctan(3/0) and incorrectly said 0 instead of pi/

In [60]:
df_ood_gen.iloc[0]['prompt']

array([{'content': 'You are a helpful case generator and an expert in mathematical reasoning. You help with augmenting text in the way the user specifies.', 'role': 'system'},
       {'content': "You're given a question and a correct student answer, your task is to inject an arithmetic or algebraic error (e.g., adding, subtracting, multiplying, or simplifying incorrectly) if and only if the answer allows for a reasonable augmentation. IMPORTANT: Stay as close as possible to the correct answer and refrain from explicitly stating errors in the augmented response, e.g. writing 'I incorretly calculate ...' or '... (this is a calculation error) ...'.In case it is unreasonable to augment the correct answer with an arithmetic or algebraic error (some answers do not contain arithmetic operations or algebraic manipulations), just return '#### Not applicable'. Otherwise return the complete error-augmented answer, pre-pended by a '####'. Abstract example: If the Correct Answer is [reasoning] [cor

In [90]:
def filter_parsed_ood_samples(df : pd.DataFrame) -> pd.DataFrame:
    """
        We'll apply a simple heuristic to decide whether an answer is a good augmentation:
        1) It must contain a '####' marker signaling the start of the answer.
        2) it must contain "\\boxed{" within the last 100 characters
    """
    def mapper(responses : list[str]) -> bool | str:
        r = responses[0] # only a single response per row
        try:
            i = r.rindex('####')
        except ValueError:
            return False
        if "boxed{" not in r[i:]:
            return False
        return r[i:].removeprefix('####').lstrip()

    df['parsed_response'] = df['responses'].apply(mapper)
    df = df[df['parsed_response'].astype(bool)]
    return df

df_ood_gen_parsed = filter_parsed_ood_samples(df_ood_gen.copy())

In [85]:
def input_from_prompt(prompt : str) -> str:
    qprefix = "\nQuestion:\n"
    qindex = prompt.rindex(qprefix)
    aprefix = "\nSolution:\n"
    aindex = prompt.rindex(aprefix)
    question = prompt[qindex:aindex].removeprefix(qprefix)
    return question

def ground_truth_from_prompt(prompt : str) -> str:
    aprefix = "\nSolution:\n"
    aindex = prompt.rindex(aprefix)
    gt = prompt[aindex:].removeprefix(aprefix).removesuffix('\n')
    return gt

In [92]:
a = df_ood_gen_parsed.iloc[0]['prompt'][1]['content']
input_from_prompt(a), ground_truth_from_prompt(a)

('Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$',
 'The point $(0,3)$ in rectangular coordinates is on the positive y-axis. The distance from the origin to the point is 3, and the angle from the positive x-axis to the y-axis is $\\frac{\\pi}{2}$. Therefore, the polar coordinates of the point are \\boxed{(3, \\frac{\\pi}{2})}.')

In [93]:
rows = []
for i, row in df_ood_gen_parsed.iterrows():
    prompt = row['prompt'][1]['content']
    inp, gt = input_from_prompt(prompt), ground_truth_from_prompt(prompt)
    error = row['parsed_response']
    correct = {
        'prompt' : [{'role' : 'user', 'content' : inp}, 
                    {'role' : 'assistant', 'content' : gt}],
        'behaviour_type' : 'anchor',
        'step' : 0,
        "uid" : i
    }
    incorrect = {
        'prompt' : [{'role' : 'user', 'content' : inp}, 
                    {'role' : 'assistant', 'content' : error}],
        'behaviour_type' : 'calculation_error',
        'step' : 0,
        "uid" : i
    }
    rows.append(correct); rows.append(incorrect)

In [103]:
ood_dataframe = pd.DataFrame(rows)
ood_dataframe.head(1)

,prompt,behaviour_type,step,uid
0,"[{'role': 'user', 'content': 'Convert the poin...",anchor,0,132


In [106]:
# at the time of writing this, I have to make the number of lines in the deltas dataframe I'm passing divisible by 8.
# It is what it is.

len(ood_dataframe)

2142

In [107]:
drop_last = len(ood_dataframe) % 8
ood_dataframe = ood_dataframe[:-drop_last]
assert len(ood_dataframe) % 8 == 0

In [108]:
path = "/u/rfechner/data/ariadne"
os.makedirs(path, exist_ok=True)

with open(os.path.join(path, 'ood-deltas.jsonl'), 'w') as file:
    ood_dataframe.to_json(path_or_buf=file, lines=True, orient='records')